In [0]:
VOLUME_PATH   = "/Volumes/workspace/ev_streaming/raw_events"
CHECKPOINT    = "/Volumes/workspace/ev_streaming/raw_events/_checkpoint_bronze"
BRONZE_TABLE  = "workspace.ev_streaming.bronze_charging_events"
print("Paths set ✅")

Paths set ✅


In [0]:
from pyspark.sql.functions import current_timestamp

stream = (
    spark.readStream
        .format("cloudFiles")                     # Auto Loader
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", CHECKPOINT)
        .option("cloudFiles.maxFilesPerTrigger", 10)
        .load(VOLUME_PATH)
        .withColumn("ingested_at", current_timestamp())   # audit column
)

query = (
    stream.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)               # process all new files, then stop
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()                          # wait for the batch to finish
print("Bronze streaming ingestion complete ✅")

Bronze streaming ingestion complete ✅


In [0]:
bronze = spark.read.table(BRONZE_TABLE)
print(f"Bronze rows: {bronze.count()}")   # ~125 (5 files × 25 events)
bronze.display()

Bronze rows: 125


city,cost_inr,duration_min,energy_kwh,event_time,session_id,station_id,status,_rescued_data,ingested_at
Chennai,375.63,80,37.25,2026-07-02T06:50:08.070675,65ee92cb-dec4-4e63-8970-0ea001ed842d,STN-01,interrupted,null,2026-07-02T06:51:56.451Z
Delhi,240.67,76,26.99,2026-07-02T06:50:08.070711,ce259ff3-020d-4abe-b76b-46b24e392c15,STN-04,completed,null,2026-07-02T06:51:56.451Z
Chennai,545.87,74,51.35,2026-07-02T06:50:08.070725,ae8258de-feef-4c46-a829-f16fdae8e68d,STN-01,completed,null,2026-07-02T06:51:56.451Z
Mumbai,239.03,60,27.52,2026-07-02T06:50:08.070736,482372e7-afa7-4df3-a08e-d1e9199e4a2c,STN-01,completed,null,2026-07-02T06:51:56.451Z
Hyderabad,29.27,33,3.55,2026-07-02T06:50:08.070747,e00f0157-9ddf-496e-8986-6d8a80e5dba5,STN-05,completed,null,2026-07-02T06:51:56.451Z
Hyderabad,442.98,117,37.58,2026-07-02T06:50:08.070758,0d63ec71-2d7e-409f-9e7d-24556ab6fa67,STN-03,interrupted,null,2026-07-02T06:51:56.451Z
Bengaluru,392.67,49,41.11,2026-07-02T06:50:08.070768,663f1e85-58cd-4a75-98ad-374ccdd46e95,STN-05,completed,null,2026-07-02T06:51:56.451Z
Chennai,513.63,55,48.24,2026-07-02T06:50:08.070778,5509b07e-152a-40de-8f31-1207435a481f,STN-02,interrupted,null,2026-07-02T06:51:56.451Z
Hyderabad,460.17,32,40.84,2026-07-02T06:50:08.070788,cce91e27-64cb-4cf1-927f-e89d113d9bdf,STN-03,completed,null,2026-07-02T06:51:56.451Z
Bengaluru,184.83,34,18.75,2026-07-02T06:50:08.070798,b48a208e-bde0-4f7f-b201-f96d938d1b75,STN-04,completed,null,2026-07-02T06:51:56.451Z
